In [ ]:
# -*- coding: utf-8 -*-
"""
Neural surrogate (MC-Dropout) + EI/PI acquisition to propose the next query point
for a 2D black-box maximization problem.

- Handles tiny datasets robustly (scaling, weight decay, early stop).
- Uncertainty via MC-Dropout (stochastic forward passes).
- Acquisition = Expected Improvement (EI). Also reports Probability of Improvement (PI).
- Prints the recommended next point with predicted mean ± std, EI, PI.

Requirements: numpy, torch, scikit-learn
"""

import math
import numpy as np
import torch
from torch import nn, optim
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -------------------------
# 1) Your observed data
# -------------------------
X = np.array([
    [0.31940389, 0.76295937],
    [0.57432921, 0.87989810],
    [0.73102363, 0.73299988],
    [0.84035342, 0.26473161],
    [0.65011406, 0.68152635],
    [0.41043714, 0.14755430],
    [0.31269116, 0.07872278],
    [0.68341817, 0.86105746],
    [0.08250725, 0.40348751],
    [0.88388983, 0.58225397],
    [0.88389000, 0.98389000],
    [0.37454000, 0.95071300],
    [0.38222400, 0.95131900],
    [0.78277800, 0.79332900],
], dtype=np.float64)

y = np.array([
    1.32267704e-079, 1.03307824e-046, 7.71087511e-016, 3.34177101e-124,
   -3.60606264e-003, -2.15924904e-054, -2.08909327e-091, 2.53500115e-040,
    3.60677119e-081, 6.22985647e-048, 9.59033053e-135,-1.56227724e-117,
   -4.77166224e-115, 7.535209723645751e-36
], dtype=np.float64).reshape(-1, 1)

# Sanity: best observed so far (we are maximizing)
best_idx = int(np.argmax(y))
best_y  = float(y[best_idx, 0])

# -------------------------
# 2) Preprocess
# -------------------------
# Inputs already ~[0,1], but we still standardize for NN training stability.
x_scaler = StandardScaler()
Xn = x_scaler.fit_transform(X)

# Targets span tiny positives + negatives near zero; standardize, don't log.
y_scaler = StandardScaler()
yn = y_scaler.fit_transform(y)

# -------------------------
# 3) Define MC-Dropout MLP
# -------------------------
class MCDropoutMLP(nn.Module):
    def __init__(self, in_dim=2, hidden=64, p_drop=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(p_drop),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MCDropoutMLP(in_dim=2, hidden=64, p_drop=0.15).to(device)

# -------------------------
# 4) Train with early stopping
# -------------------------
Xtr, Xva, ytr, yva = train_test_split(Xn, yn, test_size=0.25, random_state=42)
Xtr_t = torch.tensor(Xtr, dtype=torch.float32, device=device)
ytr_t = torch.tensor(ytr, dtype=torch.float32, device=device)
Xva_t = torch.tensor(Xva, dtype=torch.float32, device=device)
yva_t = torch.tensor(yva, dtype=torch.float32, device=device)

opt = optim.AdamW(model.parameters(), lr=3e-3, weight_decay=1e-3)
loss_fn = nn.MSELoss()

best_state = None
best_val = float("inf")
patience = 200
stale = 0
max_epochs = 5000

for epoch in range(max_epochs):
    model.train()  # enable dropout
    opt.zero_grad()
    pred = model(Xtr_t)
    loss = loss_fn(pred, ytr_t)
    loss.backward()
    opt.step()

    # Validation (also with dropout active to match MC behavior)
    model.train()
    with torch.no_grad():
        val = loss_fn(model(Xva_t), yva_t).item()

    if val < best_val - 1e-6:
        best_val = val
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        stale = 0
    else:
        stale += 1
        if stale > patience:
            break

# Load best weights
if best_state is not None:
    model.load_state_dict(best_state)
model.eval()  # we'll toggle to train() during MC passes

# -------------------------
# 5) Acquisition utilities
# -------------------------
@torch.no_grad()
def mc_predict(Xcand_t, T=100):
    """
    MC-dropout predictions: returns mean and std in ORIGINAL y-scale.
    """
    model.train()  # keep dropout ON for MC
    preds = []
    for _ in range(T):
        out = model(Xcand_t)                      # in normalized y-scale
        preds.append(out.detach().cpu().numpy())
    preds = np.stack(preds, axis=0).squeeze(-1)   # [T, N]
    mu_n = preds.mean(axis=0)                     # normalized mean
    std_n = preds.std(axis=0) + 1e-12            # normalized std (avoid div0)

    # Inverse-transform: for a standard scaler, y = std*y_n + mean
    mu = y_scaler.inverse_transform(mu_n.reshape(-1,1)).ravel()
    std = std_n * float(y_scaler.scale_[0])      # std scales by scaler.scale_
    return mu, std

def expected_improvement(mu, std, best_y, xi=1e-6):
    """
    EI for maximization.
    mu, std in ORIGINAL scale. best_y is best observed (original scale).
    """
    from scipy.stats import norm
    imp = mu - best_y - xi
    Z = np.divide(imp, std, out=np.zeros_like(imp), where=std>0)
    ei = imp * norm.cdf(Z) + std * norm.pdf(Z)
    ei[std < 1e-15] = 0.0
    return ei

def prob_improvement(mu, std, best_y, xi=1e-6):
    from scipy.stats import norm
    Z = np.divide(mu - best_y - xi, std, out=np.zeros_like(mu), where=std>0)
    pi = norm.cdf(Z)
    pi[std < 1e-15] = 0.0
    return pi

# -------------------------
# 6) Candidate generation (bounded to [0,1]^2)
# -------------------------
def sample_candidates(n_base=5000, n_refine=40, refine_each=200):
    """
    - Random-Latin-like base sampling across [0,1]^2
    - Gradient-free local refinement around top EI seeds
    """
    base = np.random.rand(n_base, 2)  # [0,1]^2
    # softly bias toward known good zones by mixing observed points
    if len(X) > 0:
        mix = X[np.random.randint(0, len(X), size=n_base)] + 0.05*np.random.randn(n_base,2)
        mix = np.clip(mix, 0.0, 1.0)
        base = np.vstack([base, mix])

    # Local refinement: jitter around top seeds
    seeds = base[np.random.choice(len(base), size=refine_each, replace=False)]
    jitters = seeds[:, None, :] + 0.05*np.random.randn(refine_each, n_refine, 2)
    jitters = np.clip(jitters, 0.0, 1.0).reshape(-1, 2)
    return np.vstack([base, jitters])

# -------------------------
# 7) Score candidates and select the best
# -------------------------
def propose_next_point(T_mc=200):
    C = sample_candidates(n_base=5000, n_refine=50, refine_each=300)
    Cn = x_scaler.transform(C)
    Cn_t = torch.tensor(Cn, dtype=torch.float32, device=device)

    mu, std = mc_predict(Cn_t, T=T_mc)     # original scale
    ei = expected_improvement(mu, std, best_y=best_y, xi=1e-9)
    pi = prob_improvement(mu, std, best_y=best_y, xi=1e-9)

    idx = int(np.argmax(ei))
    x_next = C[idx]
    out = {
        "x_next": x_next,
        "pred_mean": float(mu[idx]),
        "pred_std":  float(std[idx]),
        "EI": float(ei[idx]),
        "PI": float(pi[idx]),
        "best_seen_y": float(best_y),
    }
    return out

rec = propose_next_point(T_mc=300)

# -------------------------
# 8) Report
# -------------------------
print("\n=== Bayesian Next-Query Recommendation (NN + MC-Dropout) ===")
print(f"Best observed y*: {rec['best_seen_y']:.6e}")
print(f"Proposed x_next : [{rec['x_next'][0]:.6f}, {rec['x_next'][1]:.6f}]  (in original [0,1] bounds)")
print(f"Predicted y_mean: {rec['pred_mean']:.6e}  ± {rec['pred_std']:.2e} (MC std)")
print(f"ExpectedImprovement (EI): {rec['EI']:.6e}")
print(f"ProbabilityImprovement (PI): {rec['PI']:.3f}")

# Brief reasoning breadcrumb (prints to console so you can log it)
print("\nReasoning:")
print("- Fitted an MLP surrogate with MC-Dropout to capture epistemic uncertainty.")
print("- Evaluated EI across a large candidate pool + local jitter, then chose the argmax.")
print("- PI gives the probability this point beats the current best; EI balances mean/uncertainty.")
